#  분자 특징화(Featurization) 더 깊이 알아보기

분자 데이터로 머신러닝을 수행할 때 가장 중요한 단계 중 하나는, 데이터를 학습 알고리즘을 적용하기 좋은 형태로 변환하는 것입니다. 이 과정을 넓은 의미에서 "특징화(featurization)"라고 부르며, 분자를 어떤 형태의 벡터(vector)나 텐서(tensor)로 바꾸는 작업을 포함합니다. 특징화에는 여러 가지 방법이 있으며, 어떤 방법을 선택할지는 다루려는 문제에 따라 달라지는 경우가 많습니다. 우리는 이미 두 가지 방법을 살펴보았습니다. 바로 분자 지문(molecular fingerprint)과, 그래프 합성곱(graph convolution)에 사용되는 `ConvMol` 객체입니다. 이번 튜토리얼에서는 그 외의 다른 방법들을 살펴보겠습니다.

## Kaggle에서 실행하기

이 노트북은 Kaggle에서 실행할 수 있습니다. 아래 배지를 클릭하면 Kaggle에서 바로 열립니다.

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/isg-yhlee93/lecture/blob/main/Molecular%20Machine%20Learning/2_Going_Deeper_on_Molecular_Featurizations.ipynb)


In [1]:
!pip install -qq --pre deepchem
import deepchem

# 불필요한 경고 메시지 출력 안하기
import warnings
warnings.filterwarnings('ignore')

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

deepchem.__version__


'2.6.0.dev'

## 특징화 도구(Featurizer)

DeepChem에서 분자(또는 그 밖의 어떤 입력)를 특징화하는 방법은 `Featurizer` 객체로 정의됩니다. 특징화 도구를 사용하는 방법은 세 가지가 있습니다.

1. MoleculeNet 로더 함수를 사용할 때는, 사용할 특징화 방법의 이름을 그냥 넘겨주면 됩니다. `featurizer='ECFP'`나 `featurizer='GraphConv'`처럼, 앞선 튜토리얼에서 이미 예시를 본 적이 있습니다.

2. 직접 `Featurizer`를 생성해 분자에 바로 적용할 수도 있습니다. 예를 들면 다음과 같습니다.


In [2]:
import deepchem as dc

# 원형 지문(Circular Fingerprint) 특징화 도구를 만들어 SMILES 문자열들에 직접 적용합니다.
featurizer = dc.feat.CircularFingerprint()
print(featurizer(['CC', 'CCC', 'CCO']))


[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


[09:05:59] DEPRECATION WARNING: please use MorganGenerator
[09:05:59] DEPRECATION WARNING: please use MorganGenerator
[09:05:59] DEPRECATION WARNING: please use MorganGenerator


3. DataLoader 프레임워크로 새로운 데이터셋을 만들 때, 데이터 처리에 사용할 `Featurizer`를 지정할 수 있습니다. 이 방법은 이후 튜토리얼에서 살펴보겠습니다.

이번 튜토리얼에서는 프로페인(propane, CH<sub>3</sub>CH<sub>2</sub>CH<sub>3</sub>, SMILES 문자열로는 `'CCC'`)을 계속해서 예시로 사용합니다. 많은 특징화 방법은 분자의 형태이성질체(conformer)를 사용합니다. 형태이성질체는 `deepchem.utils.conformers`의 `ConformerGenerator` 클래스로 생성할 수 있습니다.


### RDKitDescriptors


`RDKitDescriptors`는 RDKit을 사용하여 여러 분자 기술자(descriptor) 값을 계산함으로써 분자를 특징화합니다. 이 기술자들은 분자량(molecular weight), 극성 표면적(polar surface area), 수소 결합 주개(donor)·받개(acceptor)의 개수 등 기본적인 물리·화학적 성질입니다. 이는 세밀한 분자 구조보다는 이러한 상위 수준의 성질에 의존하는 값을 예측할 때 가장 유용합니다.

이 특징화 도구는 내부적으로 허용된 기술자(descriptor) 집합을 가지고 있으며, `RDKitDescriptors.allowedDescriptors`로 확인할 수 있습니다. 특징화 도구는 `rdkit.Chem.Descriptors.descList`에 있는 기술자들을 사용하되, 그것들이 허용된 기술자 목록에 있는지 확인한 뒤 분자에 대한 기술자 값을 계산합니다.

프로페인에 대해 처음 10개 기술자의 값을 출력해 봅시다.


In [3]:
from rdkit.Chem.Descriptors import descList
rdkit_featurizer = dc.feat.RDKitDescriptors()
features = rdkit_featurizer(['CCC'])[0]
descriptors = [i[0] for i in descList]
# 기술자 이름과 계산된 값을 짝지어 처음 10개를 출력합니다.
for feature, descriptor in zip(features[:10], descriptors):
    print(descriptor, feature)


MaxAbsEStateIndex 2.125
MaxEStateIndex 2.125
MinAbsEStateIndex 1.25
MinEStateIndex 1.25
qed 0.3854706587740357
SPS 6.0
MolWt 44.097
HeavyAtomMolWt 36.033
ExactMolWt 44.062600255999996
NumValenceElectrons 20.0


[09:07:17] DEPRECATION WARNING: please use MorganGenerator
[09:07:17] DEPRECATION WARNING: please use MorganGenerator
[09:07:17] DEPRECATION WARNING: please use MorganGenerator


물론 이보다 훨씬 더 많은 기술자가 있습니다.


In [4]:
print('존재하는 기술자(descriptor)의 개수: ', len(features))


The number of descriptors present is:  210


### WeaveFeaturizer와 MolGraphConvFeaturizer

앞에서 우리는 `ConvMolFeaturizer`를 사용해 분자를 `ConvMol` 객체로 변환하는 그래프 합성곱(graph convolution)을 살펴보았습니다. 그래프 합성곱은 분자를 그래프(graph)로 표현하는 더 큰 구조 부류(class)의 한 특수한 경우입니다. 이들은 비슷한 방식으로 동작하지만 세부적인 부분에서 차이가 있습니다. 예를 들어, 데이터 벡터를 원자(atom)에 연관시킬 수도, 원자를 잇는 결합(bond)에 연관시킬 수도, 혹은 둘 다에 연관시킬 수도 있습니다. 또한 이전 층의 데이터로부터 새로운 데이터 벡터를 계산하는 기법도 다양하고, 마지막에 분자 수준(molecule-level)의 성질을 계산하는 기법도 다양합니다.

DeepChem은 여러 종류의 그래프 기반 모델을 지원합니다. 그중 일부는 분자를 조금씩 다른 방식으로 특징화해야 합니다. 이 때문에 `WeaveFeaturizer`와 `MolGraphConvFeaturizer`라는 두 가지 특징화 도구가 추가로 존재합니다. 이들은 각각 분자를 특정 모델에서 사용하는 서로 다른 종류의 파이썬 객체로 변환합니다. 그래프 기반 모델을 사용할 때는 해당 모델과 함께 어떤 특징화 도구를 써야 하는지 문서를 꼭 확인하세요.


### CoulombMatrix (쿨롱 행렬)

지금까지 살펴본 모델들은 분자를 구성하는 원자 목록과 그것들을 잇는 결합처럼, 분자의 내재적(intrinsic) 성질만을 고려했습니다. 그런데 유연한(flexible) 분자를 다룰 때는, 그 분자가 취할 수 있는 다양한 형태(conformation)도 함께 고려하고 싶을 수 있습니다. 예를 들어 약물 분자가 단백질에 결합할 때, 결합의 세기는 특정 원자 쌍 사이의 구체적인 상호작용에 따라 달라집니다. 결합 세기를 예측하려면 다양한 가능한 형태를 고려하고, 예측 시 이를 반영하는 모델을 사용하는 것이 좋습니다.

쿨롱 행렬(Coulomb matrix)은 분자의 형태(conformation)를 표현하는 대표적인 특징화 방법 중 하나입니다. 두 전하 사이의 정전기적 쿨롱 상호작용은 $q_1 q_2/r$에 비례한다는 것을 떠올려 보세요. 여기서 $q_1$, $q_2$는 전하이고 $r$은 두 전하 사이의 거리입니다. 원자가 $N$개인 분자에 대해, 쿨롱 행렬은 $N \times N$ 크기의 행렬이며 각 원소는 두 원자 사이의 정전기적 상호작용 세기를 나타냅니다. 이 행렬은 원자의 전하 정보와 원자 사이의 거리 정보를 모두 담고 있습니다. 사용된 함수 형태에 대한 자세한 정보는 [여기](https://journals.aps.org/prl/pdf/10.1103/PhysRevLett.108.058301)에서 확인할 수 있습니다.

이 특징화 도구를 적용하려면 먼저 분자에 대한 형태(conformation) 집합이 필요합니다. 이를 위해 `ConformerGenerator` 클래스를 사용할 수 있습니다. 이 클래스는 RDKit 분자를 입력받아 에너지가 최소화된 형태이성질체(conformer) 집합을 생성하고, 서로 충분히 다른 것들만 남도록 집합을 정리(prune)합니다. 프로페인에 대해 실행해 봅시다.


In [5]:
from rdkit import Chem

# 최대 5개의 형태이성질체(conformer)를 생성하는 생성기를 만듭니다.
generator = dc.utils.ConformerGenerator(max_conformers=5)
propane_mol = generator.generate_conformers(Chem.MolFromSmiles('CCC'))
print("프로페인에서 사용 가능한 형태이성질체 수: ", len(propane_mol.GetConformers()))


Number of available conformers for propane:  1


형태이성질체를 단 하나만 찾았습니다. 프로페인은 매우 작아 거의 유연성이 없는 분자이므로 놀랄 일은 아닙니다. 이번에는 탄소를 하나 더 추가해 봅시다.


In [6]:
butane_mol = generator.generate_conformers(Chem.MolFromSmiles('CCCC'))
print("뷰테인(butane)에서 사용 가능한 형태이성질체 수: ", len(butane_mol.GetConformers()))


Number of available conformers for butane:  3


이제 분자에 대한 쿨롱 행렬을 만들 수 있습니다.


In [7]:
# 최대 원자 수를 20으로 지정하여 쿨롱 행렬 특징화 도구를 만듭니다.
coulomb_mat = dc.feat.CoulombMatrix(max_atoms=20)
features = coulomb_mat(propane_mol)
print(features)


[[[36.8581052  12.48684429  7.5619687   2.85945193  2.85804514
    2.85804556  1.4674015   1.46740144  0.91279491  1.14239698
    1.14239675  0.          0.          0.          0.
    0.          0.          0.          0.          0.        ]
  [12.48684429 36.8581052  12.48684388  1.46551218  1.45850736
    1.45850732  2.85689525  2.85689538  1.4655122   1.4585072
    1.4585072   0.          0.          0.          0.
    0.          0.          0.          0.          0.        ]
  [ 7.5619687  12.48684388 36.8581052   0.9127949   1.14239695
    1.14239692  1.46740146  1.46740145  2.85945178  2.85804504
    2.85804493  0.          0.          0.          0.
    0.          0.          0.          0.          0.        ]
  [ 2.85945193  1.46551218  0.9127949   0.5         0.29325367
    0.29325369  0.21256978  0.21256978  0.12268391  0.13960187
    0.13960185  0.          0.          0.          0.
    0.          0.          0.          0.          0.        ]
  [ 2.85804514  1.458

많은 원소가 0인 것을 확인할 수 있습니다. 여러 분자를 하나의 배치(batch)로 묶으려면, 분자마다 원자 수가 다르더라도 모든 쿨롱 행렬의 크기가 같아야 합니다. 우리는 `max_atoms=20`으로 지정했으므로 반환된 행렬의 크기는 (20, 20)입니다. 이 분자는 원자가 11개뿐이므로, 11×11 부분 행렬(submatrix)만 0이 아닌 값을 가집니다.


### CoulombMatrixEig (쿨롱 행렬 고윳값)


쿨롱 행렬의 중요한 특성 중 하나는 분자의 회전(rotation)과 평행이동(translation)에 대해 불변(invariant)이라는 점입니다. 원자 간 거리와 원자 번호(atomic number)는 회전·평행이동으로 바뀌지 않기 때문입니다. 이러한 대칭성을 존중하면 학습이 더 쉬워집니다. 분자를 회전시켜도 물리적 성질은 바뀌지 않습니다. 만약 특징화 결과가 회전에 따라 바뀐다면 모델은 "회전은 중요하지 않다"는 사실을 따로 학습해야 하지만, 특징화가 불변이라면 모델은 이 성질을 자동으로 얻게 됩니다.

하지만 쿨롱 행렬은 또 다른 중요한 대칭성, 즉 원자 인덱스(index)의 치환(permutation)에 대해서는 불변이 아닙니다. 어떤 원자를 "1번 원자"라고 부르든 분자의 물리적 성질은 달라지지 않지만, 쿨롱 행렬은 달라집니다. 이를 해결하기 위해 `CoulombMatrixEig` 특징화 도구가 도입되었습니다. 이는 쿨롱 행렬의 고윳값 스펙트럼(eigenvalue spectrum)을 사용하므로 원자 인덱스의 임의 치환에 대해 불변입니다. 이 특징화의 단점은 담고 있는 정보가 훨씬 적다는 점입니다($N \times N$ 행렬 대신 $N$개의 고윳값). 따라서 모델이 학습할 수 있는 내용에 더 많은 제약이 생깁니다.

`CoulombMatrixEig`는 `CoulombMatrix`를 상속받으며, 먼저 분자의 여러 형태이성질체에 대해 쿨롱 행렬을 계산한 다음 각 쿨롱 행렬의 고윳값을 계산하는 방식으로 분자를 특징화합니다. 이렇게 얻은 고윳값들은 분자마다 다른 원자 수를 맞추기 위해 패딩(padding)됩니다.


In [8]:
coulomb_mat_eig = dc.feat.CoulombMatrixEig(max_atoms=20)
features = coulomb_mat_eig(propane_mol)
print(features)


[[60.07620303 29.62963149 22.75497781  0.5713786   0.28781332  0.28548338
   0.27558187  0.18163794  0.17460999  0.17059719  0.16640098  0.
   0.          0.          0.          0.          0.          0.
   0.          0.        ]]


### SMILES 토큰화(Tokenization)와 수치화(Numericalization)


지금까지 우리는 SMILES 데이터에 암묵적으로 담긴 구조적·물리적 정보를, 모델이 학습하고 예측하는 데 도움이 되는 더 명시적인 특징으로 변환하는 특징화 기법들을 살펴보았습니다. 이번 절에서는 SMILES 문자열을 1차원 합성곱 신경망(1D CNN)이나 트랜스포머(transformer) 같은 시퀀스 모델(sequence model)에 입력할 수 있는 형태로 전처리합니다. 이를 통해 이러한 모델이 분자 성질에 대한 표현(representation)을 스스로 학습할 수 있게 됩니다.

SMILES 문자열을 시퀀스 모델에 맞게 준비하기 위해, 우리는 SMILES를 부분 문자열(토큰, token) 목록으로 분해한 뒤, 이를 정수 값 목록으로 변환합니다(수치화, numericalization). 시퀀스 모델은 이 정수 값들을 임베딩 행렬(embedding matrix)의 인덱스로 사용합니다. 임베딩 행렬은 어휘(vocabulary)에 있는 각 토큰마다 부동소수점(floating-point) 숫자로 이루어진 벡터를 담고 있습니다. 이 임베딩 벡터들은 모델 학습 중에 갱신됩니다. 이 과정을 통해 시퀀스 모델은 학습 데이터에 암묵적으로 담긴 분자 성질에 대한 표현을 스스로 학습할 수 있습니다.

SMILES 토큰화 과정을 보여주기 위해 DeepChem의 `BasicSmilesTokenizer`와 MoleculeNet의 Tox21 데이터셋을 사용하겠습니다.


In [9]:
import numpy as np


In [10]:
# 원본(Raw) 형태로 Tox21 데이터셋을 불러옵니다(특징화하지 않은 상태).
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer="Raw")
train_dataset, valid_dataset, test_dataset = datasets
print(train_dataset)


<DiskDataset X.shape: (6264,), y.shape: (6264, 12), w.shape: (6264, 12), task_names: ['NR-AR' 'NR-AR-LBD' 'NR-AhR' ... 'SR-HSE' 'SR-MMP' 'SR-p53']>


데이터셋을 `featurizer="Raw"`로 불러왔습니다. 이제 각 데이터셋의 `ids` 속성에서 SMILES 문자열을 얻습니다.


In [11]:
train_smiles = train_dataset.ids
valid_smiles = valid_dataset.ids
test_smiles = test_dataset.ids
print(train_smiles[:5])


['CC(O)(P(=O)(O)O)P(=O)(O)O' 'CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C'
 'OC[C@H](O)[C@@H](O)[C@H](O)CO'
 'CCCCCCCC(=O)[O-].CCCCCCCC(=O)[O-].[Zn+2]' 'CC(C)COC(=O)C(C)C']


다음으로 토크나이저(tokenizer)를 정의하고 이를 모든 데이터에 `map`으로 적용하여 SMILES 문자열을 토큰 목록으로 변환합니다. `BasicSmilesTokenizer`는 SMILES를 대략 원자(atom) 수준으로 분해합니다.


In [12]:
tokenizer = dc.feat.smiles_tokenizer.BasicSmilesTokenizer()

# 모든 SMILES 문자열을 토큰 목록으로 변환합니다.
train_tok = list(map(tokenizer.tokenize, train_smiles))
valid_tok = list(map(tokenizer.tokenize, valid_smiles))
test_tok = list(map(tokenizer.tokenize, test_smiles))
print(train_tok[0])
len(train_tok)


['C', 'C', '(', 'O', ')', '(', 'P', '(', '=', 'O', ')', '(', 'O', ')', 'O', ')', 'P', '(', '=', 'O', ')', '(', 'O', ')', 'O']


6264

이제 데이터셋의 모든 SMILES 문자열을 토큰화한 버전을 얻었습니다. 이것을 정수 값 목록으로 변환하려면 먼저 데이터셋에 등장하는 모든 토큰의 목록을 만들어야 합니다. 이 목록을 어휘(vocabulary)라고 부릅니다. 또한 0으로 패딩(zero-padding)된 수치화 SMILES를 디코딩할 때 끝부분의 0을 올바르게 처리하기 위해, 빈 문자열 `""`도 어휘에 추가합니다.


In [13]:
flatten = lambda l: [item for items in l for item in items]

# 전체 데이터에 등장하는 모든 토큰을 모아 중복을 제거하고 정렬하여 어휘를 만듭니다.
all_toks = flatten(train_tok) + flatten(valid_tok) + flatten(test_tok)
vocab = sorted(set(all_toks + [""]))
print(vocab[:12], "...", vocab[-12:])
len(vocab)


['', '#', '(', ')', '-', '.', '/', '1', '2', '3', '4', '5'] ... ['[n+]', '[n-]', '[nH+]', '[nH]', '[o+]', '[s+]', '[se]', '\\', 'c', 'n', 'o', 's']


128

토큰화된 SMILES 문자열을 수치화하기 위해, 어휘의 각 토큰에 번호를 부여하는 `str2int` 딕셔너리를 만듭니다. 역방향 딕셔너리인 `int2str`도 함께 만들고, 이에 대응하는 `encode`·`decode` 함수를 정의합니다. 마지막으로 토큰화된 데이터에 `encode` 함수를 `map`으로 적용하여 수치화된 SMILES 데이터를 얻습니다.


In [14]:
# 토큰 -> 정수, 정수 -> 토큰 변환 딕셔너리를 만듭니다.
str2int = {s:i for i, s in enumerate(vocab)}
int2str = {i:s for i, s in enumerate(vocab)}
print(f"str2int: {dict(list(str2int.items())[:5])} ...")
print(f"int2str: {dict(list(int2str.items())[:5])} ...")


str2int: {'': 0, '#': 1, '(': 2, ')': 3, '-': 4} ...
int2str: {0: '', 1: '#', 2: '(', 3: ')', 4: '-'} ...


In [15]:
encode = lambda s: [str2int[tok] for tok in s]   # 토큰 목록 -> 정수 목록
decode = lambda i: [int2str[num] for num in i]   # 정수 목록 -> 토큰 목록
print(train_smiles[0])
print(encode(train_tok[0]))
print("".join(decode(encode(train_tok[0]))))  # 인코딩 후 디코딩하면 원래 SMILES로 복원됩니다.


CC(O)(P(=O)(O)O)P(=O)(O)O
[19, 19, 2, 24, 3, 2, 25, 2, 16, 24, 3, 2, 24, 3, 24, 3, 25, 2, 16, 24, 3, 2, 24, 3, 24]
CC(O)(P(=O)(O)O)P(=O)(O)O


In [16]:
train_num = list(map(encode, train_tok))
valid_num = list(map(encode, valid_tok))
test_num = list(map(encode, test_tok))
print(train_num[0])


[19, 19, 2, 24, 3, 2, 25, 2, 16, 24, 3, 2, 24, 3, 24, 3, 25, 2, 16, 24, 3, 2, 24, 3, 24]


마지막으로, 모델에 배치(batch) 단위로 입력할 수 있도록 데이터셋의 모든 분자를 하나의 `np.array`로 합치고자 합니다. 이를 위해서는 모든 시퀀스의 길이가 같아야 합니다. CoulombMatrix 절에서처럼, 정해진 길이까지 0을 덧붙여(append) 이를 달성합니다.


In [17]:
# 전체 데이터에서 가장 긴 시퀀스의 길이를 구합니다.
max_len = max(map(len, train_num + valid_num + test_num))
max_len


240

Tox21 전체 데이터셋에서 가장 긴 시퀀스의 길이는 `240`이므로, 이를 고정 길이로 사용합니다. `zero_pad` 함수를 만들어 모든 수치화 SMILES에 `map`으로 적용한 뒤 `np.array`로 변환합니다.


In [18]:
# 각 시퀀스 뒤에 0을 덧붙여 길이를 max_len으로 맞춥니다.
zero_pad = lambda x: x + [0] * (max_len - len(x))

train_numpad = np.array(list(map(zero_pad, train_num)))
valid_numpad = np.array(list(map(zero_pad, valid_num)))
test_numpad = np.array(list(map(zero_pad, test_num)))
train_numpad


array([[19, 19,  2, ...,  0,  0,  0],
       [19, 19,  2, ...,  0,  0,  0],
       [24, 19, 42, ...,  0,  0,  0],
       ...,
       [24, 16, 19, ...,  0,  0,  0],
       [19, 19,  2, ...,  0,  0,  0],
       [19, 19,  2, ...,  0,  0,  0]])

임의의 데이터 한 점에 대해 `decode` 함수를 적용해, 0으로 패딩된 데이터가 여전히 올바른 SMILES 문자열로 복원되는지 확인할 수 있습니다.


In [19]:
idx = np.random.randint(0, train_numpad.shape[0], 1).item()
print(train_smiles[idx])
print("".join(decode(train_numpad[idx])))


Cc1cc(C(C)(C)c2ccc(O)c(C)c2)ccc1O
Cc1cc(C(C)(C)c2ccc(O)c(C)c2)ccc1O


패딩된 데이터가 검증을 통과했습니다. 이제 시퀀스 모델 학습에 사용할 수 있는 올바른 형식이 되었지만, 아직 DeepChem의 학습 프레임워크와 매끄럽게 연동되지는 않습니다. 이를 위해, 위에서 설명한 모든 단계를 하나로 합쳐 데이터 한 점을 처리하는 `tokenize_smiles` 함수를 정의합니다. 추가로, 이 사용자 정의 `tokenize_smiles` 함수를 `_featurize` 메서드에서 사용하는 `SmilesFeaturizer`를 정의하고, 우리의 `vocab`과 `max_len`을 넘겨 `smiles_featurizer`로 인스턴스화합니다.


In [20]:
def tokenize_smiles(x, vocab, max_len):
    # 토큰화 -> 수치화(인코딩) -> 0 패딩 단계를 하나로 합친 함수입니다.
    tokenizer = dc.feat.smiles_tokenizer.BasicSmilesTokenizer()
    str2int = {s:i for i, s in enumerate(vocab)}
    encode = lambda s: [str2int[tok] for tok in s]
    zero_pad = lambda x: x + [0] * (max_len - len(x))
    x = tokenizer.tokenize(x)
    x = encode(x)
    x = zero_pad(x)
    return np.array(x)

class SmilesFeaturizer(dc.feat.Featurizer):
    # 위 함수를 사용하는 사용자 정의 특징화 도구입니다.
    def __init__(self, feat_func, vocab, max_len):
        self.feat_func = feat_func
        self.vocab = vocab
        self.max_len = max_len
        
    def _featurize(self, x):
        return self.feat_func(x, self.vocab, self.max_len)
    
smiles_featurizer = SmilesFeaturizer(tokenize_smiles, vocab, max_len)


마지막으로 `smiles_featurizer`를 사용하여, `X` 속성에 토큰화·수치화된 SMILES를 담은 새로운 Tox21 데이터셋을 생성합니다.


In [21]:
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer=smiles_featurizer)
print(datasets[0].X)


[09:24:48] WARNING: not removing hydrogen atom without neighbors


[[19 19  2 ...  0  0  0]
 [19 19  2 ...  0  0  0]
 [24 19 42 ...  0  0  0]
 ...
 [24 16 19 ...  0  0  0]
 [19 19  2 ...  0  0  0]
 [19 19  2 ...  0  0  0]]


이제 데이터셋은 여러분의 사용자 정의 DeepChem 시퀀스 모델과 함께 사용할 준비가 되었습니다. 모델을 적절한 DeepChem 모델 클래스로 감싸는 것을 잊지 마세요.


# 축하합니다! 이제 커뮤니티에 참여할 시간입니다!

이 튜토리얼 노트북을 완료하신 것을 축하합니다! 튜토리얼을 즐겁게 따라오셨고 DeepChem을 계속 활용하고 싶으시다면, 이 시리즈의 나머지 튜토리얼도 마저 완료해 보시길 권장합니다. 또한 다음과 같은 방법으로 DeepChem 커뮤니티에 도움을 주실 수 있습니다.

## [GitHub](https://github.com/deepchem/deepchem)에서 DeepChem에 Star 누르기
이는 DeepChem 프로젝트와, 우리가 만들고자 하는 오픈소스 신약 개발 도구에 대한 관심을 높이는 데 도움이 됩니다.

## DeepChem Gitter 참여하기
DeepChem [Gitter](https://gitter.im/deepchem/Lobby)에는 생명과학 분야의 딥러닝에 관심이 있는 많은 과학자, 개발자, 애호가들이 모여 있습니다. 대화에 함께 참여해 보세요!


## 이 튜토리얼 인용하기
이 튜토리얼이 유용했다면, 아래 제공된 BibTeX를 사용해 인용해 주시길 부탁드립니다.


In [ ]:
@manual{Intro7, 
 title={Going Deeper on Molecular Featurizations}, 
 organization={DeepChem},
 author={Ramsundar, Bharath}, 
 howpublished = {\url{https://github.com/deepchem/deepchem/blob/master/examples/tutorials/Going_Deeper_on_Molecular_Featurizations.ipynb}}, 
 year={2021}, 
} 